## Phishing Website Detection: Exhaustive End-to-End Pipeline

### 1. Problem Statement
Phishing is a cybercrime in which targets are contacted by malicious actors posing as legitimate institutions to lure individuals into providing sensitive data. 

The objective of this project is to build an exhaustive, highly efficient classification model capable of detecting whether a given data point constitutes a phishing attack. We will leverage the `phising-data.csv` dataset to perform rigorous Exploratory Data Analysis (EDA), advanced feature engineering (handling multicollinearity, zero-variance features, and mutual information selection), and train multiple classification algorithms to deploy the best solution.

In [ ]:
## 1. Importing Important Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

### 2. Data Loading & Basic Inspection

In [ ]:
# Load the dataset
df = pd.read_csv('phising-data.csv')
print(f'Dataset Shape: {df.shape}')
display(df.head())

In [ ]:
# Check data types and overall information
df.info()
display(df.describe().T)

### 3. Exploratory Data Analysis (EDA)
We will explore the target distribution, check for missing values, analyze feature distributions, and map correlations.

In [ ]:
## 3.1 Target Variable Analysis
# Dynamically identifying the target column (assuming it is the last column)
target_col = df.columns[-1]
print(f'Identified Target Column: {target_col}')

# Standardize target to 0 and 1 if it contains -1 (Needed for XGBoost and logic mapping)
if df[target_col].min() == -1:
    print('Converting target labels from -1/1 to 0/1 for algorithm compatibility.')
    df[target_col] = df[target_col].replace({-1: 0})

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
df[target_col].value_counts().plot.pie(autopct='%1.1f%%', explode=[0, 0.05], shadow=True, colors=['#ff9999','#66b3ff'])
plt.title('Distribution of Target Classes (Pie)')
plt.ylabel('')

plt.subplot(1, 2, 2)
sns.countplot(x=target_col, data=df, palette='viridis')
plt.title('Distribution of Target Classes (Bar)')
plt.xlabel('Class (0: Legitimate, 1: Phishing)')

plt.tight_layout()
plt.show()

In [ ]:
## 3.2 Missing Values Check
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0]
if missing_data.empty:
    print('No missing values found in the dataset! Proceeding cleanly.')
else:
    print('Missing values found:')
    display(missing_data)
    missing_data.plot(kind='bar', color='coral', figsize=(10, 4))
    plt.title('Missing Values per Feature')
    plt.show()

In [ ]:
## 3.3 Feature Distributions
# Plotting histograms for all features to see their variances and distributions
df.drop(columns=[target_col]).hist(bins=15, figsize=(20, 18), color='teal', edgecolor='black')
plt.suptitle('Independent Feature Distributions', fontsize=20, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
## 3.4 Correlation Heatmap
plt.figure(figsize=(22, 18))
corr_matrix = df.corr()
# Create a mask for the upper triangle to remove visual clutter
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=22)
plt.show()

In [ ]:
## 3.5 Correlation with Target
plt.figure(figsize=(14, 6))
target_corr = df.corr()[target_col].drop(target_col).sort_values(ascending=False)
target_corr.plot(kind='bar', color=np.where(target_corr > 0, 'dodgerblue', 'crimson'))
plt.title('Correlation of Independent Features with the Target', fontsize=16)
plt.axhline(0, color='black', linewidth=1)
plt.ylabel('Pearson Correlation Coefficient')
plt.show()

### 4. Advanced Feature Engineering & Selection
1. **Variance Threshold:** Remove constant features that provide no predictive power.
2. **Multicollinearity Removal:** Drop features that are highly correlated with each other to reduce redundancy.
3. **Mutual Information:** Measure nonlinear dependencies between independent variables and the target.

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

## 4.1 Remove Constant Features (Zero Variance)
var_thres = VarianceThreshold(threshold=0)
var_thres.fit(X)
constant_columns = [col for col in X.columns if col not in X.columns[var_thres.get_support()]]

print(f'Number of constant features detected: {len(constant_columns)}')
if len(constant_columns) > 0:
    print(f'Dropped features: {constant_columns}')
    X = X.drop(columns=constant_columns)

In [ ]:
## 4.2 Handling Multicollinearity
# Identify highly correlated features to drop (Threshold: > 0.85)
corr_features = X.corr().abs()
upper = corr_features.where(np.triu(np.ones(corr_features.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

print(f'Number of highly correlated features dropped: {len(to_drop)}')
print(f'Features dropped due to collinearity: {to_drop}')

X = X.drop(columns=to_drop)
print(f'Shape of X after dropping highly correlated features: {X.shape}')

In [ ]:
## 4.3 Feature Importance via Mutual Information
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_scores_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(14, 6))
mi_scores_series.plot(kind='bar', color='mediumseagreen')
plt.title('Feature Importance via Mutual Information Scores', fontsize=16)
plt.ylabel('Mutual Information Score')
plt.axhline(0.01, color='red', linestyle='--', label='Relevance Threshold (0.01)')
plt.legend()
plt.show()

# Drop features with extremely low mutual info (< 0.01) if necessary
low_mi_features = mi_scores_series[mi_scores_series < 0.01].index
print(f'Features with very low predictive power (MI < 0.01): {list(low_mi_features)}')
# Optional: X = X.drop(columns=low_mi_features) 

### 5. Data Preprocessing (Train-Test Split & Scaling)

In [ ]:
# Split dataset into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training data shape: X={X_train.shape}, y={y_train.shape}')
print(f'Testing data shape: X={X_test.shape}, y={y_test.shape}')

# Standardize the features for distance-based algorithms to perform optimally
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 6. Baseline Model Training & Evaluation
We will test a suite of algorithms to establish a baseline performance before tuning.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boost': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []

for name, model in models.items():
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    # Evaluate metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred, average='weighted')
    prec = precision_score(y_test, y_test_pred)
    rec = recall_score(y_test, y_test_pred)
    
    if hasattr(model, 'predict_proba'):
        roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    else:
        roc_auc = np.nan
        
    results.append({
        'Model': name, 'Train Acc': train_acc, 'Test Acc': test_acc, 
        'F1 Score': f1, 'Precision': prec, 'Recall': rec, 'ROC AUC': roc_auc
    })

    print(f'=========== {name} ===========')
    print(f'- Train Accuracy : {train_acc:.4f}')
    print(f'- Test Accuracy  : {test_acc:.4f}')
    print(f'- F1 Score       : {f1:.4f}')
    print(f'- Precision      : {prec:.4f}')
    print(f'- Recall         : {rec:.4f}')
    print(f'- ROC AUC        : {roc_auc:.4f}\n')

In [ ]:
## Visualizing Baseline Performance
results_df = pd.DataFrame(results).sort_values(by='Test Acc', ascending=False)
display(results_df)

plt.figure(figsize=(10, 5))
sns.barplot(x='Test Acc', y='Model', data=results_df, palette='magma')
plt.title('Baseline Model Comparison by Test Accuracy')
plt.xlim(0.8, 1.0)
plt.show()

### 7. Hyperparameter Tuning
We will tune the top-performing ensemble models (Random Forest and XGBoost) to combat overfitting and maximize unseen accuracy.

In [ ]:
## 7.1 Tuning Random Forest
print('Starting Hyperparameter Tuning for Random Forest...')

rf_params = {
    'n_estimators': [100, 200, 400, 600],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf_random = RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42), 
                               param_distributions=rf_params, 
                               n_iter=20, cv=3, verbose=1, random_state=42, n_jobs=-1)

rf_random.fit(X_train_scaled, y_train)
best_rf = rf_random.best_estimator_

print(f'\nBest Parameters for Random Forest:\n{rf_random.best_params_}')
print(f'Tuned RF Test Accuracy: {accuracy_score(y_test, best_rf.predict(X_test_scaled)):.4f}')

In [ ]:
## 7.2 Tuning XGBoost
print('Starting Hyperparameter Tuning for XGBoost...')

xgb_params = {
    'n_estimators': [100, 200, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'subsample': [0.6, 0.8, 1.0]
}

xgb_random = RandomizedSearchCV(estimator=XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42), 
                                param_distributions=xgb_params, 
                                n_iter=20, cv=3, verbose=1, random_state=42, n_jobs=-1)

xgb_random.fit(X_train_scaled, y_train)
best_xgb = xgb_random.best_estimator_

print(f'\nBest Parameters for XGBoost:\n{xgb_random.best_params_}')
print(f'Tuned XGBoost Test Accuracy: {accuracy_score(y_test, best_xgb.predict(X_test_scaled)):.4f}')

### 8. Final Evaluation & Results Plotting

In [ ]:
## 8.1 ROC-AUC Curve
plt.figure(figsize=(10, 8))

# Calculate Probabilities
rf_probs = best_rf.predict_proba(X_test_scaled)[:, 1]
xgb_probs = best_xgb.predict_proba(X_test_scaled)[:, 1]

# Calculate FPR, TPR, Thresholds
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)

# Plot the lines
plt.plot(rf_fpr, rf_tpr, label=f'Tuned Random Forest (AUC = {roc_auc_score(y_test, rf_probs):.4f})', linewidth=2)
plt.plot(xgb_fpr, xgb_tpr, label=f'Tuned XGBoost (AUC = {roc_auc_score(y_test, xgb_probs):.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Chance')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves for Top Tuned Models', fontsize=16)
plt.legend(loc='lower right', fontsize=12)
plt.show()

In [ ]:
## 8.2 Confusion Matrices
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_estimator(best_rf, X_test_scaled, y_test, ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title('Tuned Random Forest Confusion Matrix')
ax[0].grid(False)

ConfusionMatrixDisplay.from_estimator(best_xgb, X_test_scaled, y_test, ax=ax[1], cmap='Oranges', colorbar=False)
ax[1].set_title('Tuned XGBoost Confusion Matrix')
ax[1].grid(False)

plt.tight_layout()
plt.show()